# Five-class provenance attribution: end-to-end demo
This notebook runs Algorithms 1, 2 and 4 of the manuscript on synthetic logs of one invented student (`synthetic_logs.json`, built by `make_synthetic.py`). No study data are needed. Classes: C chatbot-first, E echo (entered through the student first), S shared source, T typed, X untraced insertion (source not recorded in the logs).

In [ ]:
import json
from provenance import attribute, shares, editor_entry, segment_uptake, ngrams
logs = json.load(open('synthetic_logs.json'))
print(logs['turns'])
print(logs['replies'][0][1][:120], '...')

## Attribute every token
`n` is the n-gram length, `tau` the largest edit (in tokens) still counted as typing, `delta` the clock tolerance.

In [ ]:
A, classes, label, u_star = attribute(logs['final'], logs['replies'], logs['turns'], logs['snapshots'], logs['shared'], n=8, tau=5, delta=0)
for tok, c in list(zip(A, classes))[:20]:
    print(c, tok)
print(shares(classes), 'U* =', round(u_star, 3))

## What changes with the parameters
A larger `tau` turns the untraced paste into typed text; a smaller `n` matches shorter phrases.

In [ ]:
for n in (5, 8, 12):
    for tau in (5, 50):
        _, cl, _, u = attribute(logs['final'], logs['replies'], logs['turns'], logs['snapshots'], logs['shared'], n=n, tau=tau)
        print(f'n={n:2d} tau={tau:2d}', {k: round(v, 2) for k, v in shares(cl).items()})

## Direction matters
Ignoring the student's earlier prompt turns the echoed draft into chatbot-first text.

In [ ]:
_, cl, _, u = attribute(logs['final'], logs['replies'], [], logs['snapshots'], logs['shared'])
print('without prompts:', {k: round(v, 2) for k, v in shares(cl).items()}, 'U* =', round(u, 3))

## Segment uptake (Algorithm 4)

In [ ]:
t_ed, _ = editor_entry(ngrams(A, 8), logs['snapshots'], 8)
print(segment_uptake([tuple(s) for s in logs['segments']], logs['replies'], label, t_ed))